# Optimizer Deep Dive: From Gradient Descent to Adam

A from-scratch study of optimization algorithms for machine learning, implemented entirely in NumPy.

**Author:** Chris Schmidt — MS Applied Mathematics | AI Engineering MSE (JHU)

## Contents
1. Motivation & Setup
2. Mathematical Background
3. Batch Gradient Descent
4. SGD with Momentum
5. Adam Optimizer
6. Optimizer Showdown on Test Functions
7. Scale to Real Data (MNIST)
8. Neural Network from Scratch
9. Initialization Sensitivity
10. Loss Landscape Visualization
11. Conclusions

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.optimizers import BatchGD, SGD, Adam, minimize, mse_loss, mse_grad, softmax
from src.models import NeuralNetwork, train_network, relu, sigmoid
from src.visualization import (
    plot_optimization_traces,
    plot_2d_contour,
    plot_grad_norms,
    plot_training_history,
    plot_loss_landscape,
    plot_initialization_comparison,
)

EVIDENCE = Path('../evidence')
EVIDENCE.mkdir(exist_ok=True)

np.random.seed(42)
print('Setup complete.')

## 2. Mathematical Background

All gradient-based optimizers follow the same template:

$$\theta_{t+1} = \theta_t - \eta \cdot g(\nabla f(\theta_t))$$

where $g(\cdot)$ transforms the raw gradient. The three algorithms differ in how they compute $g$:

| Algorithm | Update Rule | Key Idea |
|-----------|------------|----------|
| **Batch GD** | $\theta \leftarrow \theta - \eta \nabla f$ | Full gradient, deterministic |
| **SGD + Momentum** | $v_t = \mu v_{t-1} + \nabla f$; $\theta \leftarrow \theta - \eta v_t$ | Exponential averaging accelerates consistent directions |
| **Adam** | $m_t, v_t$ = bias-corrected 1st/2nd moments; $\theta \leftarrow \theta - \eta \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$ | Adaptive per-parameter learning rates |

### Key Questions
- How does curvature (condition number) affect convergence?
- When does momentum help vs. hurt?
- Why does Adam dominate in practice for deep networks?

## 3. Batch Gradient Descent

We start with the simplest optimizer on well-known test functions.

In [ ]:
# Rosenbrock function: f(x,y) = (1-x)^2 + 100(y-x^2)^2
def rosenbrock(params):
    x, y = params
    loss = (1 - x)**2 + 100 * (y - x**2)**2
    grad = np.array([
        -2*(1 - x) - 400*x*(y - x**2),
        200*(y - x**2)
    ])
    return loss, grad

# Quadratic bowl with different curvatures: f(x) = 0.5 * x^T A x
def make_quadratic(condition_number=10):
    eigenvalues = np.array([1.0, condition_number])
    def f(params):
        loss = 0.5 * np.sum(eigenvalues * params**2)
        grad = eigenvalues * params
        return loss, grad
    return f

# Test BGD on quadratic
x0 = np.array([4.0, 4.0])
quadratic = make_quadratic(condition_number=50)

trace_bgd = minimize(quadratic, x0, BatchGD(lr=0.01), n_steps=200)
print(f'BGD final loss: {trace_bgd.loss[-1]:.6f}')
print(f'BGD converged to: {trace_bgd.params[-1]}')

## 4. SGD with Momentum

Momentum helps accelerate through elongated valleys by accumulating velocity.

In [ ]:
# Compare BGD vs SGD vs SGD+Momentum on the ill-conditioned quadratic
x0 = np.array([4.0, 4.0])
quadratic = make_quadratic(condition_number=50)

traces = {
    'BGD (lr=0.01)': minimize(quadratic, x0, BatchGD(lr=0.01), n_steps=200),
    'SGD (lr=0.01)': minimize(quadratic, x0, SGD(lr=0.01, momentum=0.0), n_steps=200),
    'SGD+Mom (lr=0.01, μ=0.9)': minimize(quadratic, x0, SGD(lr=0.01, momentum=0.9), n_steps=200),
}

for name, t in traces.items():
    print(f'{name}: final loss = {t.loss[-1]:.8f}')

In [ ]:
# 2D trajectory comparison on the quadratic
fig = plot_2d_contour(quadratic, traces, xlim=(-5, 5), ylim=(-5, 5),
                      title='Optimizer Trajectories (Ill-conditioned Quadratic κ=50)')
plt.show()

## 5. Adam Optimizer

Adam combines momentum (first moment) with RMSProp-style adaptive learning rates (second moment).

In [ ]:
x0 = np.array([4.0, 4.0])

traces_adam = {
    'BGD (lr=0.01)': minimize(quadratic, x0, BatchGD(lr=0.01), n_steps=300),
    'SGD+Mom (lr=0.01, μ=0.9)': minimize(quadratic, x0, SGD(lr=0.01, momentum=0.9), n_steps=300),
    'Adam (lr=0.01)': minimize(quadratic, x0, Adam(lr=0.01), n_steps=300),
    'Adam (lr=0.001)': minimize(quadratic, x0, Adam(lr=0.001), n_steps=300),
}

fig = plot_optimization_traces(traces_adam, title='Convergence Comparison (κ=50)')
plt.show()

for name, t in traces_adam.items():
    print(f'{name}: final loss = {t.loss[-1]:.10f}')

## 6. Optimizer Showdown on Test Functions

Systematic comparison across Rosenbrock and quadratics with varying condition numbers.

In [ ]:
# Rosenbrock comparison
x0_rosen = np.array([-1.0, 1.0])

rosen_traces = {
    'BGD (lr=0.001)': minimize(rosenbrock, x0_rosen, BatchGD(lr=0.001), n_steps=5000),
    'SGD+Mom (lr=0.001, μ=0.9)': minimize(rosenbrock, x0_rosen, SGD(lr=0.001, momentum=0.9), n_steps=5000),
    'Adam (lr=0.001)': minimize(rosenbrock, x0_rosen, Adam(lr=0.001), n_steps=5000),
}

fig = plot_2d_contour(rosenbrock, rosen_traces, xlim=(-2, 2), ylim=(-1, 3),
                      title='Rosenbrock Valley (minimum at (1,1))')
fig.savefig(EVIDENCE / 'rosenbrock_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_optimization_traces(rosen_traces, title='Rosenbrock Convergence')
fig.savefig(EVIDENCE / 'optimizer_showdown.png', dpi=150, bbox_inches='tight')
plt.show()

for name, t in rosen_traces.items():
    final_p = t.params[-1]
    print(f'{name}: final = ({final_p[0]:.4f}, {final_p[1]:.4f}), loss = {t.loss[-1]:.6f}')

In [ ]:
# Gradient norm analysis
fig = plot_grad_norms(rosen_traces, title='Gradient Norms on Rosenbrock')
fig.savefig(EVIDENCE / 'gradient_norms.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Condition number sensitivity study
import pandas as pd

results = []
for kappa in [2, 10, 50, 100, 500]:
    f = make_quadratic(kappa)
    x0 = np.array([4.0, 4.0])
    for name, opt in [('BGD', BatchGD(lr=0.001)),
                       ('SGD+Mom', SGD(lr=0.001, momentum=0.9)),
                       ('Adam', Adam(lr=0.01))]:
        trace = minimize(f, x0, opt, n_steps=500)
        results.append({
            'κ': kappa,
            'Optimizer': name,
            'Final Loss': trace.loss[-1],
            'Steps to 1e-4': next((i for i, l in enumerate(trace.loss) if l < 1e-4), 'N/A'),
        })

df = pd.DataFrame(results)
print(df.to_string(index=False))

## 7. Scale to Real Data (MNIST)

Load MNIST and train a from-scratch neural network using each optimizer.

In [ ]:
from sklearn.datasets import fetch_openml

# Load MNIST (sklearn provides a small version)
X_raw, y_raw = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
X_raw = X_raw / 255.0  # normalize to [0, 1]
y_raw = y_raw.astype(int)

# Use a subset for speed
n_train, n_test = 5000, 1000
X_train, y_train = X_raw[:n_train], y_raw[:n_train]
X_test, y_test = X_raw[60000:60000+n_test], y_raw[60000:60000+n_test]

print(f'Training: {X_train.shape}, Test: {X_test.shape}')
print(f'Labels: {np.unique(y_train)}')

## 8. Neural Network from Scratch

Train a 784 → 128 → 10 network with each optimizer and compare convergence.

In [ ]:
histories = {}
test_accuracies = {}

configs = [
    ('SGD (lr=0.01)', lambda n: [SGD(lr=0.01) for _ in range(n)]),
    ('SGD+Mom (lr=0.01)', lambda n: [SGD(lr=0.01, momentum=0.9) for _ in range(n)]),
    ('Adam (lr=0.001)', lambda n: [Adam(lr=0.001) for _ in range(n)]),
]

for name, opt_factory in configs:
    print(f'Training with {name}...')
    net = NeuralNetwork([784, 128, 10], activation='relu', task='classification', seed=42)
    hist = train_network(net, X_train, y_train, opt_factory, epochs=30, batch_size=64)
    histories[name] = hist
    test_acc = net.accuracy(X_test, y_test)
    test_accuracies[name] = test_acc
    print(f'  Final train acc: {hist["accuracy"][-1]:.4f}, Test acc: {test_acc:.4f}')

fig = plot_training_history(histories, title='MNIST Neural Net: Optimizer Comparison')
fig.savefig(EVIDENCE / 'nn_training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table
summary = pd.DataFrame([
    {'Optimizer': name,
     'Final Train Loss': hist['loss'][-1],
     'Final Train Acc': hist['accuracy'][-1],
     'Test Acc': test_accuracies[name]}
    for name, hist in histories.items()
])
print(summary.to_string(index=False))

## 9. Initialization Sensitivity

How does weight initialization affect training? We compare He, Xavier, and uniform strategies.

In [ ]:
init_results = {}

# He initialization (default for ReLU)
net_he = NeuralNetwork([784, 128, 10], activation='relu', task='classification', seed=42)
init_results['He (ReLU)'] = train_network(
    net_he, X_train, y_train,
    lambda n: [Adam(lr=0.001) for _ in range(n)],
    epochs=30, batch_size=64
)

# Xavier initialization (Sigmoid)
net_xavier = NeuralNetwork([784, 128, 10], activation='sigmoid', task='classification', seed=42)
init_results['Xavier (Sigmoid)'] = train_network(
    net_xavier, X_train, y_train,
    lambda n: [Adam(lr=0.001) for _ in range(n)],
    epochs=30, batch_size=64
)

# Large random initialization (intentionally bad)
net_large = NeuralNetwork([784, 128, 10], activation='relu', task='classification', seed=42)
for i in range(net_large.n_layers):
    net_large.weights[i] *= 5.0  # scale up 5x
init_results['Large Random (5x)'] = train_network(
    net_large, X_train, y_train,
    lambda n: [Adam(lr=0.001) for _ in range(n)],
    epochs=30, batch_size=64
)

fig = plot_initialization_comparison(init_results)
fig.savefig(EVIDENCE / 'initialization_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

for name, hist in init_results.items():
    print(f'{name}: final loss={hist["loss"][-1]:.4f}, acc={hist["accuracy"][-1]:.4f}')

## 10. Loss Landscape Visualization

Visualize the loss surface around the trained solution using random direction projections.

In [ ]:
# Train a fresh network for landscape visualization
net_viz = NeuralNetwork([784, 128, 10], activation='relu', task='classification', seed=42)
train_network(net_viz, X_train, y_train,
              lambda n: [Adam(lr=0.001) for _ in range(n)],
              epochs=30, batch_size=64)

# Generate random directions (normalized)
rng = np.random.default_rng(123)
base_params = net_viz.get_params()
dir1 = [rng.standard_normal(p.shape) for p in base_params]
dir2 = [rng.standard_normal(p.shape) for p in base_params]

# Normalize to same scale as parameters
for i in range(len(dir1)):
    norm1 = np.linalg.norm(dir1[i])
    norm2 = np.linalg.norm(dir2[i])
    p_norm = np.linalg.norm(base_params[i])
    if norm1 > 0:
        dir1[i] = dir1[i] / norm1 * p_norm
    if norm2 > 0:
        dir2[i] = dir2[i] / norm2 * p_norm

# Use a small subset for speed
X_sub, y_sub = X_train[:500], y_train[:500]

fig = plot_loss_landscape(net_viz, X_sub, y_sub, dir1, dir2,
                          n_points=20, alpha_range=(-0.5, 0.5))
fig.savefig(EVIDENCE / 'loss_landscape.png', dpi=150, bbox_inches='tight')
plt.show()
print('Loss landscape rendered.')

## 11. Conclusions

### Key Findings

1. **Batch GD** converges reliably but slowly, especially on ill-conditioned problems (high κ)
2. **Momentum** dramatically accelerates convergence through elongated valleys by accumulating consistent gradient directions
3. **Adam** achieves the best balance — adaptive per-parameter learning rates handle heterogeneous curvature automatically
4. **Initialization matters**: He initialization for ReLU networks, Xavier for sigmoid — wrong initialization can prevent learning entirely
5. **The loss landscape** around trained solutions is typically bowl-shaped but can have complex curvature that explains why adaptive methods excel

### Mathematical Insight

The convergence rate of gradient descent on a quadratic is governed by $\kappa = \lambda_{\max} / \lambda_{\min}$:

$$\|\theta_t - \theta^*\| \leq \left(\frac{\kappa - 1}{\kappa + 1}\right)^t \|\theta_0 - \theta^*\|$$

Adam effectively reduces the effective condition number by rescaling each coordinate independently.